In [1]:
import random
import os
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import torchvision.transforms as transforms
import torchvision.models as models
import requests


from PIL import Image


from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score
from transformers import AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig
from torch.utils.data import Dataset, DataLoader,TensorDataset
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor

In [2]:
# ==========================================
# 2-a Read-CSV Drop empty url
# ==========================================

df = pd.read_csv('movies.csv', engine='python')
os.makedirs('./posters', exist_ok=True)

# Drop rows where 'Poster_Url' is missing (NaN)
df.dropna(subset=['Poster_Url','Genre'], inplace=True)

# Save the cleaned dataset back to a new CSV (optional but recommended)
df.to_csv('movies_cleaned.csv', index=False)

In [3]:
# ==========================================
# 2-B Download movie posters in multi-threads
# ==========================================

# Add a movie_id column based on the index since it doesn't exist in the CSV
df['movie_id'] = df.index

def download_image(row):
    # Fix column name to match the DataFrame
    url = row['Poster_Url']
    movie_id = row['movie_id']
    save_path = f"./posters/{movie_id}.jpg"

    if pd.isna(url):
        return

    # Skip if already downloaded
    if os.path.exists(save_path):
        return

    try:
        response = requests.get(url, timeout=5)
        if response.status_code == 200:
            with open(save_path, 'wb') as f:
                f.write(response.content)
    except Exception:
        pass # Handle or log errors silently to keep threads moving

# Use ThreadPoolExecutor to download 32 images at a time
records = df.to_dict('records')
with ThreadPoolExecutor(max_workers=32) as executor:
    executor.map(download_image, records)

In [4]:
# ==========================================
# 2-C Sanity Check on the movie poster folder
# ==========================================
poster_dir = './posters'
if os.path.exists(poster_dir):
    downloaded_files = os.listdir(poster_dir)
    num_downloaded = len(downloaded_files)
    total_movies = len(df)

    print(f"Total posters downloaded: {num_downloaded} out of {total_movies}")

    if num_downloaded == total_movies:
        print("All posters have been downloaded successfully!")
    else:
        print(f"Still missing {total_movies - num_downloaded} posters.")
else:
    print("The './posters' directory does not exist yet.")

Total posters downloaded: 9826 out of 9826
All posters have been downloaded successfully!


In [5]:
# ==========================================
# 2-D Split dataset into 5 equal parts
# ==========================================

# Encode multi-label genres into a binary matrix
mlb = MultiLabelBinarizer()
df['labels'] = mlb.fit_transform(df['Genre'].apply(lambda x: x.split(', '))).tolist()
valid_genres = ", ".join(mlb.classes_)

# Fix random seed for reproducibility
SEED = 50
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Shuffle the dataframe using random seed
df_shuffled = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

# Split into 5 nearly equal datasets
datasets = np.array_split(df_shuffled, 5)

# Assign to separate variables for easy access later
df_1, df_2, df_3, df_4, df_5 = datasets

# Verify the sizes of each new dataset
for i, d in enumerate(datasets, 1):
    print(f"Dataset {i} size: {len(d)}")

# Combine datasets 1-4 for training, use dataset 5 for testing
df_train = pd.concat([df_1, df_2, df_3, df_4]).reset_index(drop=True)
df_test = df_5.reset_index(drop=True)

Dataset 1 size: 1966
Dataset 2 size: 1965
Dataset 3 size: 1965
Dataset 4 size: 1965
Dataset 5 size: 1965


/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


In [6]:
# ==========================================
# 3-A Create Dataset
# ==========================================

class MoviePosterDataset(Dataset):
    def __init__(self, dataframe, img_dir, transform=None):
        self.dataframe = dataframe
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        movie_id = self.dataframe.loc[idx, 'movie_id']
        img_path = os.path.join(self.img_dir, f"{movie_id}.jpg")

        # Open image and convert to RGB (some might be grayscale)
        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        labels = torch.tensor(self.dataframe.loc[idx, 'labels'], dtype=torch.float32)
        return image, labels

# Standard ImageNet transformations
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = MoviePosterDataset(df_train, './posters', transform=transform)
test_dataset = MoviePosterDataset(df_test, './posters', transform=transform)

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=2)

print(f"Training samples: {len(train_dataset)}")
print(f"Testing samples: {len(test_dataset)}")

Training samples: 7861
Testing samples: 1965


In [7]:
# ==========================================
# 3-B Training
# ==========================================

# Initialize pre-trained ResNet-18
num_classes = len(df_train.iloc[0]['labels'])
model = models.resnet18(pretrained=True)

# Modify the final fully connected layer to output our number of classes
model.fc = nn.Linear(model.fc.in_features, num_classes)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

# Multi-label classification requires Binary Cross Entropy with Logits
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

print(f"Model loaded and sent to: {device}")

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Model loaded and sent to: cuda


In [8]:
num_epochs = 10

print("Starting Training...")
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]"):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1} Average Loss: {running_loss/len(train_loader):.4f}")

Starting Training...


Epoch 1/10 [Train]: 100%|██████████| 123/123 [03:45<00:00,  1.83s/it]


Epoch 1 Average Loss: 0.3543


Epoch 2/10 [Train]: 100%|██████████| 123/123 [03:42<00:00,  1.81s/it]


Epoch 2 Average Loss: 0.2396


...

Epoch 8 Average Loss: 0.0229


Epoch 9/10 [Train]: 100%|██████████| 123/123 [03:37<00:00,  1.77s/it]


Epoch 9 Average Loss: 0.0162


Epoch 10/10 [Train]: 100%|██████████| 123/123 [03:35<00:00,  1.75s/it]

Epoch 10 Average Loss: 0.0124


In [9]:
# ==========================================
# 3-C Test
# ==========================================

# Evaluation
model.eval()
all_preds = []
all_targets = []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Evaluating"):
        images = images.to(device)
        outputs = model(images)

        # Apply sigmoid to convert logits to probabilities, then threshold at 0.5
        preds = torch.sigmoid(outputs) > 0.5

        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(labels.numpy())

all_preds = np.array(all_preds)
all_targets = np.array(all_targets)

# Calculate evaluation metrics
# Macro F1 is useful for minority classes since it calculates F1 per class and averages them
macro_f1 = f1_score(all_targets, all_preds, average='macro', zero_division=0)
micro_f1 = f1_score(all_targets, all_preds, average='micro', zero_division=0)
subset_acc = accuracy_score(all_targets, all_preds)

print("\n=== Final Results on Test Set ===")
print(f"Micro F1-Score:  {micro_f1:.4f}")
print(f"Subset Accuracy (Exact Match): {subset_acc:.4f}")

Evaluating: 100%|██████████| 31/31 [00:55<00:00,  1.80s/it]


=== Final Results on Test Set ===
Micro F1-Score:  0.4994
Subset Accuracy (Exact Match): 0.0936
